## 1. 加载模型

对https://docs.unsloth.ai/get-started/fine-tuning-guide的内容进行总结

In [ ]:
from unsloth import FastLanguageModel

model_name = "xxx"
load_in_4bit = True
max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

- model_name
  - 以unsloth-bnb-4bit结尾:代表该模型是Unsloth dynamic 4-bit quants。比标准的BitsAndBytes 4-bit消耗更多显存但提供更高的精度
  - 以bnb-4bit结尾：代表标准的BitsAndBytes 4-bit quantization
  - 没有后缀的模型是原始的16位或8位格式

- other setting
  - max_seq_length: 控制上下文长度
  - dtype = None：Defaults to None;
  - load_in_4bit = True：启用4位量化，减少内存使用4倍微调。禁用它允许启用LoRA 16位微调
  - 要启用完全微调（FFT），请设置full_finetuning = True。对于8位微调，设置load_in_8bit = True。注意：一次只能设置一种训练方法为True。

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

挂载lora

## 2. Dataset

- 需要使用map将数据集处理为model期望的格式
- 可能需要使用chat template，暂时还没有遇到。

## 3. Training + Evaluation

use hugging face library

## 4. Running + Saving the model

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")